In [1]:
from sentence_transformers import (
  CrossEncoder,
  InputExample,
  losses,
  evaluation,
  SentenceTransformer,
  models,
  SentenceTransformerTrainer,
  SentenceTransformerTrainingArguments
  )
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator, CEBinaryAccuracyEvaluator
from datasets import Dataset
import math
import json
import torch
from torch.utils.data import DataLoader
from datetime import datetime
import random
import math

In [2]:
data_ratio = 1

with open('data/train.json', 'r') as f:
    train = json.load(f)
random.shuffle(train)

with open('data/cv.json', 'r') as f:
    cv = json.load(f)
random.shuffle(cv)

with open('data/test.json', 'r') as f:
    test = json.load(f)
random.shuffle(cv)

In [3]:
train_batch_size = 128
eval_batch_size = 64
num_epochs = 10
warmup_steps = math.ceil(len(train) * num_epochs * 0.1)

## CrossEncoder Model

In [4]:
CE_model = CrossEncoder(
  "cross-encoder/ms-marco-MiniLM-L-6-v2",
  device="mps" if torch.backends.mps.is_available() else "cpu",
  default_activation_function=torch.nn.Sigmoid()
  )

In [5]:
inputs = [InputExample(texts=x, label=y) for [x, y] in train[0:math.floor(len(train) * data_ratio)]]
dev = [InputExample(texts=x, label=y) for [x, y] in cv[0:math.floor(len(cv) * data_ratio)]]

In [ ]:
CE_model_save_path = "output/training_CE_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
CE_model.fit(
    train_dataloader=DataLoader(
        dataset=inputs,
        shuffle=True,
        batch_size=train_batch_size,
        pin_memory=True),
    evaluator=CEBinaryClassificationEvaluator.from_input_examples(dev),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=CE_model_save_path
    )

with open(CE_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size))
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")

In [ ]:
print(CE_model.predict([
  ("place of origin", "birthplace"),
  ("alien", "foreigner"),
  ("warship", "naval war vessel"),
  ("TikTok influencer", "someone who's famous on tiktok"),
  ("lemon treats", "lemon-flavored candy"),
  ("conflict", "drama"),
  ("clothing worn on the feet","socks"),
  ("throughout","always"),
  ("a break", "vacation")
  ]))
print(CE_model.predict([
  ("ice cream", "fondue"),
  ("warship", "Statue of Liberty"),
  ("birdfeed", "the study of birds"),
  ("staff made of magic", "jazz-fusion"),
  ("to be betrayed","betrothed"),
  ]))

[0.9910773  0.9964958  0.96390057 0.9408352  0.9721844  0.99073505  0.9788255  0.9946185  0.9773716 ]

[0.8128979  0.17632882 0.4739063  0.2846621  0.36774132]

## SentenceTransformer Model

In [4]:
""" transformer, pooling, norm = SentenceTransformer("all-MiniLM-L6-v2")
dense = models.Dense(in_features=pooling.word_embedding_dimension, out_features=1, activation_function=torch.nn.Sigmoid())

ST_model = SentenceTransformer(
  modules=[transformer, pooling, dense, norm],
  device="mps" if torch.backends.mps.is_available() else "cpu"
) """
ST_model = SentenceTransformer(
  "all-MiniLM-L6-v2",
  device="mps" if torch.backends.mps.is_available() else "cpu"
)

loss = losses.OnlineContrastiveLoss(ST_model)

print(ST_model)


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [5]:
train_dict = {
  "sentence1": [x[0] for [x, y] in train[0:math.floor(len(train) * data_ratio)]],
  "sentence2": [x[1] for [x, y] in train[0:math.floor(len(train) * data_ratio)]],
  "label": [y for [x, y] in train[0:math.floor(len(train) * data_ratio)]]
}
train_data = Dataset.from_dict(train_dict)

cv_dict = {
  "sentence1": [x[0] for [x, y] in cv[0:math.floor(len(train) * data_ratio)]],
  "sentence2": [x[1] for [x, y] in cv[0:math.floor(len(train) * data_ratio)]],
  "label": [y for [x, y] in cv[0:math.floor(len(train) * data_ratio)]]
}
cv_data = Dataset.from_dict(cv_dict)

test_dict = {
  "sentence1": [x[0] for [x, y] in test[0:math.floor(len(train) * data_ratio)]],
  "sentence2": [x[1] for [x, y] in test[0:math.floor(len(train) * data_ratio)]],
  "label": [y for [x, y] in test[0:math.floor(len(train) * data_ratio)]]
}
test_data = Dataset.from_dict(cv_dict)

In [6]:
ST_model_save_path = "output/training_ST_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
args = SentenceTransformerTrainingArguments(
    output_dir=ST_model_save_path,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    warmup_ratio=0.1
)

In [7]:
dev_evaluator = evaluation.BinaryClassificationEvaluator(
    sentences1=cv_data["sentence1"],
    sentences2=cv_data["sentence2"],
    labels=cv_data["label"],
    name="st-dev",
)
dev_evaluator(ST_model)

{'st-dev_cosine_accuracy': 0.8576511956963337,
 'st-dev_cosine_accuracy_threshold': 0.13289546966552734,
 'st-dev_cosine_f1': 0.9173629550234261,
 'st-dev_cosine_f1_threshold': 0.12835469841957092,
 'st-dev_cosine_precision': 0.8688322874369386,
 'st-dev_cosine_recall': 0.9716359633280011,
 'st-dev_cosine_ap': 0.8990046306190045,
 'st-dev_cosine_mcc': 0.44702500508745213}

{'st-dev_cosine_accuracy': 0.8144604306468107,

 'st-dev_cosine_accuracy_threshold': 1.0,

 'st-dev_cosine_f1': 0.897742363877822,

 'st-dev_cosine_f1_threshold': 1.0,

 'st-dev_cosine_precision': 0.8144692416537077,

 'st-dev_cosine_recall': 0.9999827992500473,

 'st-dev_cosine_ap': 0.8144604306468107,
 
 'st-dev_cosine_mcc': 0.0}

In [8]:
trainer = SentenceTransformerTrainer(
    model=ST_model,
    args=args,
    train_dataset=train_data,
    eval_dataset=cv_data,
    loss=loss,
    evaluator=dev_evaluator
)
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | f

Step,Training Loss


In [ ]:
test_evaluator = evaluation.BinaryClassificationEvaluator(
    sentences1=test_data["sentence1"],
    sentences2=test_data["sentence2"],
    labels=test_data["label"],
    name="st-test"
)
test_evaluator(ST_model)

In [32]:
with open(ST_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size) + "\n")
    f.write("eval batch size: " + str(eval_batch_size) + "\n")
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")
    f.write("no dense layer")

In [ ]:
positive = [
  ["place of origin", "birthplace"],
  ["alien", "foreigner"],
  ["warship", "naval war vessel"],
  ["TikTok influencer", "someone who's famous on tiktok"],
  ["lemon treats", "lemon-flavored candy"],
  ["conflict", "drama"],
  ["clothing worn on the feet","socks"],
  ["throughout","always"],
  ["a break", "vacation"]
]
negative = [
  ["ice cream", "fondue"],
  ["warship", "Statue of Liberty"],
  ["birdfeed", "the study of birds"],
  ["staff made of magic", "jazz-fusion"],
  ["to be betrayed","betrothed"]
]

for [x, y] in positive:
  similarity = ST_model.similarity(ST_model.encode(x), ST_model.encode(y))
  print(x, ", ", y, similarity)
for [x, y] in negative:
  similarity = ST_model.similarity(ST_model.encode(x), ST_model.encode(y))
  print(x, ", ", y, similarity)